# 04_als_model

In [1]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession

# 1. Point to your Hadoop bin folder
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")

# 2. Pin Python
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 3. Start Spark with the RawLocalFileSystem workaround
spark = SparkSession.builder \
    .appName("Spotify_ALS") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.home.dir", r"C:\hadoop") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .getOrCreate()

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Define the path to your split data
SPLIT_DIR = "../outputs/outputs/train_test_split"

# Load the train and test sets (Assuming they were saved as parquet or csv)
# Note: If your teammate saved them as CSV, change .parquet to .csv(header=True, inferSchema=True)
users_train = spark.read.csv(f"{SPLIT_DIR}/users_train", header=True, inferSchema=True)
users_test = spark.read.csv(f"{SPLIT_DIR}/users_test", header=True, inferSchema=True)

print(f"Training rows: {users_train.count()}")
users_train.show(5)

Training rows: 9107237
+-------+-------+-----------+--------------------+--------------------+-------------------+
|user_id|song_id|interaction|    user_id_original|          track_name|        artist_name|
+-------+-------+-----------+--------------------+--------------------+-------------------+
|      0| 196963|        1.0|00055176fea33f6e0...|Another You (Anot...|Against The Current|
|      0| 920045|        1.0|00055176fea33f6e0...|       Gone Too Soon|        Simple Plan|
|      0|1143595|        1.0|00055176fea33f6e0...|   If You Don't Know|5 Seconds Of Summer|
|      0|1730322|        1.0|00055176fea33f6e0...| One Of Those Nights|       Shawn Mendes|
|      0|1946760|        1.0|00055176fea33f6e0...|         Risk It All|          The Vamps|
+-------+-------+-----------+--------------------+--------------------+-------------------+
only showing top 5 rows


In [ ]:
from pyspark.ml.recommendation import ALS

# Initialize the ALS model using the columns your teammate already prepared
als = ALS(
    maxIter=10,                      
    regParam=0.1,                    
    userCol="user_id",         # Already an integer thanks to Notebook 02.2
    itemCol="song_id",         # Already an integer thanks to Notebook 02.2
    ratingCol="interaction",   # Your teammate renamed 'play_count' to 'interaction'
    coldStartStrategy="drop",        
    implicitPrefs=True               
)

print("Training the ALS Model... ")
model = als.fit(users_train)
print("Training Complete!")

Training the ALS Model... (This might take a minute or two)
Training Complete!


In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Generate predictions on the test dataset
print("Testing the model on unseen data...")
predictions = model.transform(users_test)

# 2. Show a sneak peek of what the model guessed vs the actual interaction
print("Sneak Peek at Predictions:")
predictions.select("user_id", "song_id", "interaction", "prediction").show(5)

# 3. Calculate RMSE (Root Mean Square Error)
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="interaction", 
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse:.4f}")

Testing the model on unseen data...
Sneak Peek at Predictions:
+-------+-------+-----------+------------+
|user_id|song_id|interaction|  prediction|
+-------+-------+-----------+------------+
|   2300|     81|        1.0| 6.606085E-4|
|   5952|     81|        1.0| 3.880219E-5|
|   5277|     81|        1.0|1.3120385E-5|
|  14536|    115|        1.0| 0.011583397|
|  12744|    115|        1.0|0.0076401522|
+-------+-------+-----------+------------+
only showing top 5 rows
Root-mean-square error (RMSE) = 1.1712


In [8]:
from pyspark.sql.functions import explode, col

# 1. INPUT: Pick a user (or multiple users) to generate a playlist for
user_to_test = spark.createDataFrame([(15,)], ["user_id"])

# 2. Get the Top 10 song recommendations for this specific user
raw_recs = model.recommendForUserSubset(user_to_test, 10)

# 3. CLEANUP: PySpark outputs a messy "Array" of predictions. We need to explode it into normal rows.
exploded_recs = raw_recs.select(
    "user_id", 
    explode("recommendations").alias("rec")
).select(
    "user_id",
    col("rec.song_id").alias("song_id"),
    col("rec.rating").alias("confidence_score")
)

# 4. TRANSLATE: Get the actual song names and artists from your original training data
# We use .distinct() so we don't get duplicates
song_dictionary = users_train.select("song_id", "track_name", "artist_name").distinct()

# Join the predictions with the real names and sort by highest confidence!
final_playlist = exploded_recs.join(song_dictionary, "song_id") \
                              .orderBy(col("confidence_score").desc())

# 5. OUTPUT: Show the generated playlist!
print("🎧 Top 10 Recommended Playlist for User 15 🎧")
final_playlist.select("track_name", "artist_name", "confidence_score").show(10, truncate=False)

🎧 Top 10 Recommended Playlist for User 15 🎧
+----------------------------+--------------------+----------------+
|track_name                  |artist_name         |confidence_score|
+----------------------------+--------------------+----------------+
|Juicy                       |The Notorious B.I.G.|0.054828092     |
|No Diggity                  |Blackstreet         |0.05477915      |
|Hold On, We're Going Home   |Drake               |0.052690968     |
|It Was A Good Day           |Ice Cube            |0.05266349      |
|Get Lucky - Radio Edit      |Daft Punk           |0.052270055     |
|Ni**as In Paris             |JAY Z               |0.051686082     |
|Billie Jean - Single Version|Michael Jackson     |0.051454064     |
|September                   |Earth, Wind & Fire  |0.05099216      |
|Regulate                    |Warren G            |0.05079434      |
|Bitch, Don’t Kill My Vibe   |Kendrick Lamar      |0.05043103      |
+----------------------------+--------------------+--------

In [9]:
from pyspark.sql.functions import col

print("🎧 What User 15 ACTUALLY Listened To (Training Data) 🎧")

# Filter the training data for only User 15, sort by what they played the most
actual_history = users_train.filter(col("user_id") == 15) \
                            .orderBy(col("interaction").desc())

# Show their top 15 most played songs (including how they are stored)
actual_history.select(
    "user_id", 
    "user_id_original", 
    "track_name", 
    "artist_name", 
    "interaction"
).show(15, truncate=False)

🎧 What User 15 ACTUALLY Listened To (Training Data) 🎧
+-------+--------------------------------+------------------------------------------------------+----------------+-----------+
|user_id|user_id_original                |track_name                                            |artist_name     |interaction|
+-------+--------------------------------+------------------------------------------------------+----------------+-----------+
|15     |002e678084db2e201d2721bf5af4e54c|Do You Wanna Dance? - 2001 - Remaster;Mono            |The Beach Boys  |2.0        |
|15     |002e678084db2e201d2721bf5af4e54c|All Day and All of the Night                          |The Kinks       |1.0        |
|15     |002e678084db2e201d2721bf5af4e54c|Celestica                                             |Crystal Castles |1.0        |
|15     |002e678084db2e201d2721bf5af4e54c|All I Want For Christmas Is You - Original Version    |Mariah Carey    |1.0        |
|15     |002e678084db2e201d2721bf5af4e54c|6PM In New York